In [ ]:
!pip install segmentation-models-pytorch
!pip install albumentations

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import os
print(os.listdir())

In [ ]:
import os
print(os.listdir())

In [ ]:
import os
print(os.listdir())

In [ ]:
import zipfile

with zipfile.ZipFile('EWS-Dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('data')

In [ ]:
import os
for root, dirs, files in os.walk('data'):
    print(root)

In [ ]:
import os
import cv2
import torch
from torch.utils.data import Dataset

class WheatDataset(Dataset):
    def __init__(self, root_dir):
        self.image_paths = []
        self.mask_paths = []

        for file in os.listdir(root_dir):
            if "_mask" not in file:
                self.image_paths.append(os.path.join(root_dir, file))
                mask_name = file.replace(".png", "_mask.png")
                self.mask_paths.append(os.path.join(root_dir, mask_name))

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        img=cv2.imread(self.image_paths[index])
        img=cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        m=cv2.imread(self.mask_paths[index], 0)
#FPN model
        # img = cv2.resize(img, (352, 352))
        # m = cv2.resize(m, (352, 352))

        img=img/255.0
        m=m/255.0

        img=torch.tensor(img).permute(2,0,1).float()
        m=torch.tensor(m).unsqueeze(0).float()

        return img, m

In [ ]:
train_dataset = WheatDataset('data/EWS-Dataset/train')
val_dataset = WheatDataset('data/EWS-Dataset/validation')

In [ ]:
import segmentation_models_pytorch as smp
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
)

In [ ]:
!pip install segmentation-models-pytorch

In [ ]:
import segmentation_models_pytorch as smp

In [ ]:
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
#training
for epoch in range(25):
    model.train()
    total_loss = 0

    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)

        preds = model(imgs)
        loss = loss_fn(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

In [ ]:
def iou(pred, mask):
    pred=(torch.sigmoid(pred) > 0.3).float()
    intersection=(pred * mask).sum()
    pred_area = pred.sum()
    mask_area = mask.sum()
    union = pred_area + mask_area - intersection
    e=1e-6
    return (intersection / (union + e)).item()

In [ ]:
import matplotlib.pyplot as plt
import random
model.eval()
index=random.sample(range(len(val_dataset)),5)
#img, mask = val_dataset[0]
for i in index:
  img,gmask= val_dataset[i]

model.eval()
with torch.no_grad():
    pred = model(img.unsqueeze(0).to(device))
pred = torch.sigmoid(pred)
threshold=0.9
pred = (pred>threshold).float()
plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.title("Image")
plt.imshow(img.permute(1,2,0))
plt.axis('off')
plt.subplot(1,3,2)
plt.title("Prediction")
plt.imshow(pred.squeeze().cpu(), cmap='gray')
plt.axis('off')

plt.subplot(1,3,3)
plt.title("Ground Truth")
plt.imshow(gmask.squeeze(), cmap='gray')
plt.axis('off')

plt.show()
# plt.savefig(f"result_{i}.png")
# plt.close()


In [ ]:
def iou_score(pred, mask):
    pred = (torch.sigmoid(pred) > 0.5).float()
    intersection = (pred * mask).sum()
    union = pred.sum() + mask.sum() - intersection
    return (intersection / (union + 1e-6)).item()

def dice_score(pred, mask):
    pred = (torch.sigmoid(pred) > 0.5).float()
    intersection = (pred * mask).sum()
    return (2 * intersection / (pred.sum() + mask.sum() + 1e-6)).item()

def pixel_accuracy(pred, mask):
    pred = (torch.sigmoid(pred) > 0.5).float()
    correct = (pred == mask).float().sum()
    total = torch.numel(pred)
    return (correct / total).item()

In [ ]:
model.eval()

In [ ]:
threshold = 0.6 #parameter=0.6
all_iou = []
all_dice = []
all_acc = []

for i in range(len(val_dataset)):
    image, gmask = val_dataset[i]
    image=image.to(device)
    gmask=gmask.to(device)

    with torch.no_grad():
        pred = model(image.unsqueeze(0).to(device))

    prob_map = torch.sigmoid(pred)
    pred_mask = (prob_map > threshold).float()

    all_iou.append(iou_score(pred, gmask))
    all_dice.append(dice_score(pred, gmask))
    all_acc.append(pixel_accuracy(pred, gmask))

avg_iou = sum(all_iou) / len(all_iou)
avg_dice = sum(all_dice) / len(all_dice)
avg_acc = sum(all_acc) / len(all_acc)


print(f"IoU: {avg_iou:.4f}")
print(f"Dice: {avg_dice:.4f}")
print(f"Accuracy: {avg_acc:.4f}")

In [ ]:
all_iou = []
all_dice = []
all_acc = []


all_precision = []
all_recall = []

threshold = 0.6

for i in range(len(val_dataset)):
    image, gmask = val_dataset[i]
    image = image.to(device)
    gmask = gmask.to(device)

    with torch.no_grad():
        pred = model(image.unsqueeze(0))

    prob_map = torch.sigmoid(pred)
    pred_mask = (prob_map > threshold).float()


    all_iou.append(iou_score(pred_mask, gmask))
    all_dice.append(dice_score(pred_mask, gmask))
    all_acc.append(pixel_accuracy(pred_mask, gmask))


    pred_flat = pred_mask.view(-1)
    mask_flat = gmask.view(-1)

    TP = (pred_flat * mask_flat).sum()
    FP = (pred_flat * (1 - mask_flat)).sum()
    FN = ((1 - pred_flat) * mask_flat).sum()

    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)

    all_precision.append(precision.item())
    all_recall.append(recall.item())

avg_iou = sum(all_iou) / len(all_iou)
avg_dice = sum(all_dice) / len(all_dice)
avg_acc = sum(all_acc) / len(all_acc)

avg_precision = sum(all_precision) / len(all_precision)
avg_recall = sum(all_recall) / len(all_recall)


print(f"Accuracy: {avg_acc:.4f}")
print(f"Precision: {avg_precision:.4f}")
print(f"Recall: {avg_recall:.4f}")
print(f"F1-Score(Dice): {avg_dice:.4f}")
print(f"IoU: {avg_iou:.4f}")